In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
happiness_2015 = pd.read_csv('../data/world_happiness.csv')
happiness_allyears = pd.read_csv('../data/world_happiness_allyears.csv')
who = pd.read_csv('../data/who_physical_activity.csv')

print("Happiness 2015 shape:", happiness_2015.shape)
print("Happiness all years shape:", happiness_allyears.shape)
print("WHO shape:", who.shape)
print()
print("=== All years columns ===")
print(happiness_allyears.columns.tolist())
print()
print("=== All years sample ===")
print(happiness_allyears.head(3))

Happiness 2015 shape: (158, 12)
Happiness all years shape: (1523, 4)
WHO shape: (194, 13)

=== All years columns ===
['Country', 'Year', 'Index', 'Rank']

=== All years sample ===
       Country  Year  Index  Rank
0  Afghanistan  2013  4.040   143
1  Afghanistan  2015  3.575   153
2  Afghanistan  2016  3.360   154


In [2]:
# Calculate average happiness score per country across all years
happiness_avg = happiness_allyears.groupby('Country')['Index'].mean().reset_index()
happiness_avg.columns = ['Country', 'Happiness Score Avg']

print("Countries with avg score:", len(happiness_avg))
print(happiness_avg.head(5))

Countries with avg score: 167
       Country  Happiness Score Avg
0  Afghanistan               3.0957
1      Albania               4.9589
2      Algeria               5.4103
3       Angola               4.2156
4    Argentina               6.2754


In [3]:
# Merge all three datasets
df = pd.merge(happiness_2015, happiness_avg, on='Country', how='inner')
df = pd.merge(df, who, on='Country', how='inner')

print("Final merged shape:", df.shape)
print()
print("Columns:", df.columns.tolist())

Final merged shape: (131, 25)

Columns: ['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Standard Error', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual', 'Happiness Score Avg', 'Total Population (2022)', 'Population Aged < 20 (%)', 'Population Aged > 60 (%)', 'Deaths Due to NCD', 'World Bank Income Group', 'GDP per capita (INT$)', 'Physical Inactivity, Adolescents Aged 11-17 (%), Male', 'Physical Inactivity, Adolescents Aged 11-17 (%), Female', 'Physical Inactivity, Adults Aged 18+ (%), Male', 'Physical Inactivity, Adults Aged 18+ (%), Female', 'Physical Inactivity, Adults Aged 70+ (%), Male', 'Physical Inactivity, Adults Aged 70+ (%), Female']


In [4]:
# Convert WHO percentage columns from string to float
who_cols = [
    'Physical Inactivity, Adolescents Aged 11-17 (%), Male',
    'Physical Inactivity, Adolescents Aged 11-17 (%), Female',
    'Physical Inactivity, Adults Aged 18+ (%), Male',
    'Physical Inactivity, Adults Aged 18+ (%), Female',
    'Physical Inactivity, Adults Aged 70+ (%), Male',
    'Physical Inactivity, Adults Aged 70+ (%), Female'
]

for col in who_cols:
    df[col] = df[col].astype(str).str.replace('%', '').str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing with column mean
df[who_cols] = df[who_cols].fillna(df[who_cols].mean())

# Create summary columns
df['Avg Inactivity Adults'] = df[[
    'Physical Inactivity, Adults Aged 18+ (%), Male',
    'Physical Inactivity, Adults Aged 18+ (%), Female'
]].mean(axis=1)

df['Avg Inactivity Adolescents'] = df[[
    'Physical Inactivity, Adolescents Aged 11-17 (%), Male',
    'Physical Inactivity, Adolescents Aged 11-17 (%), Female'
]].mean(axis=1)

print("Missing values after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print(df[['Country', 'Happiness Score', 'Happiness Score Avg', 'Avg Inactivity Adults']].head(5))

Missing values after cleaning:
GDP per capita (INT$)    2
dtype: int64

       Country  Happiness Score  Happiness Score Avg  Avg Inactivity Adults
0  Switzerland            7.587               7.5090               24.00000
1      Iceland            7.561               7.5114               28.71875
2      Denmark            7.527               7.5911               28.50000
3       Norway            7.522               7.4920               32.00000
4       Canada            7.427               7.2551               28.50000


In [6]:
df['GDP per capita (INT$)'] = pd.to_numeric(
    df['GDP per capita (INT$)'], errors='coerce'
)
df['GDP per capita (INT$)'] = df['GDP per capita (INT$)'].fillna(
    df['GDP per capita (INT$)'].mean()
)

df.to_csv('../data/merged_clean.csv', index=False)
print("Saved! Shape:", df.shape)

Saved! Shape: (131, 27)
